In [4]:
# data downloaded from X on 11/14/24

import pandas as pd

notes = pd.read_csv('~/Downloads/notes-00000.tsv', sep='\t')
status_history = pd.read_csv('~/Downloads/noteStatusHistory-00000.tsv', sep='\t')

/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_80904/3962277847.py:5: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('~/Downloads/notes-00000.tsv', sep='\t')
/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_80904/3962277847.py:6: DtypeWarning: Columns (10,19) have mixed types. Specify dtype option on import or set low_memory=False.
  status_history = pd.read_csv('~/Downloads/noteStatusHistory-00000.tsv', sep='\t')


In [5]:
# convert all numbers to strings, so that none of the numbers get rounded when I combine into one large df:

notes = notes.astype(str)
status_history = status_history.astype(str)

In [8]:
notesandstatus = notes.merge(status_history, on='noteId', how='outer')

In [10]:
columns_to_keep = [
    "noteId",
    "noteAuthorParticipantId_x",
    "createdAtMillis_x",
    "tweetId",
    "classification",
    "summary",
    "timestampMillisOfFirstNonNMRStatus",
    "firstNonNMRStatus"
]
notesandstatus = notesandstatus[columns_to_keep]
notesandstatus.to_csv('~/community_notes_data/notesandstatus.csv', index=False)

df = notesandstatus

In [12]:
len(df)

1566751

In [ ]:
%pip install langdetect

In [22]:
# Other cleaning procedures:
df = df[df['summary'].notna() & df['summary'].str.strip().ne("")]
df['summary'] = df['summary'].astype(str)

#remove rows with non-english in summary column: *this cell takes about 30 minutes to run

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False 

df = df[df['summary'].apply(is_english)]

In [26]:
keywords = [
    "activist", "amendment", "anti-corruption", "assembly", "autocracy", "authoritarian", 
    "Ballot", "biden", "campaign", "candidate", "capitalist", "caucus", "censorship", 
    "city council", "climate policy", "Clinton", "civic", "civil liberties", "council", 
    "Congress", "Congressman", "congresswoman", "conservative", "constitution", "corruption", 
    "Covid", "debate", "Democrat", "democracy", "diplomacy", "election", "equality", 
    "European Union", "executive", "federal", "foreign", "freedom", "governor", "Hamas", 
    "Harris", "healthcare", "Hezbollah", "House of Representatives", "impeachment", "immigration", 
    "infrastructure", "Iran", "Israel", "judiciary", "justice", "Kim Jong Un", "law", "left-wing", 
    "legislation", "liberal", "lobbyist", "mandate", "mayor", "Mcconnell", "military", "Movement", 
    "MP", "nation", "national security", "nationalism", "NATO", "Netanyahu", "Obama", "PAC", 
    "parliament", "party", "patriot", "patriotism", "Pelosi", "polling", "policy", "poverty", 
    "power", "president", "progressive", "public office", "Putin", "rally", "reform", "regulation", 
    "relations", "representative", "Republican", "rights", "sanctions", "Schumer", "Senate", "senator", 
    "socialism", "socialist", "sovereignty", "state", "stimulus", "Supreme court", "Surveillance", 
    "starmer", "tariffs", "taxes", "transparency", "Trump", "Ukraine", "Vance", "veto", "vote", 
    "Walz", "welfare", "White house", "Xi JinPing", "Zelensky"
]


In [28]:
# Function checks for keywords in "summary" column, then create 2 new columns: is_political and isnot_political

def find_keywords(text, keywords):
    text_lower = str(text).lower()
    found_keywords = [
        keyword for keyword in keywords 
        if f" {keyword.lower()} " in f" {text_lower} "
    ]
    return found_keywords

found_keywords_list = []

for text in df['summary']:
    found_keywords_list.append(find_keywords(text, keywords))

df['found_keywords'] = found_keywords_list

df['contains_keywords' = df['found_keywords'].str.len() > 0

political_tweet_ids = set(df[df['contains_keywords']]['tweetId'])

df['isnot_political'] = (df['found_keywords'].str.len() == 0) & (~df['tweetId'].isin(political_tweet_ids))

In [73]:
df.to_csv("~/community_notes_data/df.csv", index=False)

In [ ]:
# Next, manually checking to see if this process accurately groups the political and nonpolitical

In [36]:
# randomize df and manually check actual tweets for accuracy (are they political?)
df_rand = df.sample(frac=1).reset_index(drop=True)
df_rand_100 = df_rand.head(100)
df_rand_100.to_csv("df_rand_100.csv", index=False)

In [71]:
# creating new column is_political
df['is_political'] = df['contains_keywords'] | (~df['contains_keywords'] & ~df['isnot_political'])

In [77]:
# Move 'is_political' column 2 positions to the left
columns = df.columns.tolist()  
is_political_index = columns.index('is_political')  

new_position = max(0, is_political_index - 2)

columns.insert(new_position, columns.pop(is_political_index))
df = df[columns]

In [39]:
# Next few cells calculating how long it takes for Notes to get approved:

df['got_shown'] = (df['firstNonNMRStatus'] == 'CURRENTLY_RATED_HELPFUL').astype(int)

In [81]:
df.groupby('is_political').got_shown.mean()

is_political
False    0.121889
True     0.077660
Name: got_shown, dtype: float64

In [85]:
# Loop over each got_shown = 1 row and calculate length of time to be shown (in milliseconds)
df['time_difference'] = None

for index, row in df.iterrows():
    if row['got_shown'] == 1:
        df.at[index, 'time_difference'] = row['timestampMillisOfFirstNonNMRStatus'] - row['createdAtMillis_x']


In [95]:
df['hours_difference'] = df['time_difference'] / (1000 * 60 * 60)

In [99]:
# Calculate average hours for is_political Notes and isnot_political Notes:
df.groupby('is_political').time_difference.mean() / (1000 * 60 * 60)

is_political
False    35.609124
True     46.961492
Name: time_difference, dtype: object

In [91]:
df_100 = df.head(100)
df_100.to_csv('df_100.csv', index=False)

In [ ]:
# Preprocessing: Cells to Skip for Now - see below:

In [ ]:
# read in each notes_ratings file

import pandas as pd

notes_ratings0 = pd.read_csv('~/Downloads/ratings-00000.tsv', sep='\t')
notes_ratings1 = pd.read_csv('~/Downloads/ratings-00001.tsv', sep='\t')
notes_ratings2 = pd.read_csv('~/Downloads/ratings-00002.tsv', sep='\t')
notes_ratings3 = pd.read_csv('~/Downloads/ratings-00003.tsv', sep='\t')
notes_ratings4 = pd.read_csv('~/Downloads/ratings-00004.tsv', sep='\t')
notes_ratings5 = pd.read_csv('~/Downloads/ratings-00005.tsv', sep='\t')
notes_ratings6 = pd.read_csv('~/Downloads/ratings-00006.tsv', sep='\t')
notes_ratings7 = pd.read_csv('~/Downloads/ratings-00007.tsv', sep='\t')
notes_ratings8 = pd.read_csv('~/Downloads/ratings-00008.tsv', sep='\t')
notes_ratings9 = pd.read_csv('~/Downloads/ratings-00009.tsv', sep='\t')
notes_ratings10 = pd.read_csv('~/Downloads/ratings-00010.tsv', sep='\t')
notes_ratings11 = pd.read_csv('~/Downloads/ratings-00011.tsv', sep='\t')
notes_ratings12 = pd.read_csv('~/Downloads/ratings-00012.tsv', sep='\t')
notes_ratings13 = pd.read_csv('~/Downloads/ratings-00013.tsv', sep='\t')
notes_ratings14 = pd.read_csv('~/Downloads/ratings-00014.tsv', sep='\t')
notes_ratings15 = pd.read_csv('~/Downloads/ratings-00015.tsv', sep='\t')

In [ ]:
# This didn't work - still too big for my RAM
# Loop through the notes_ratings one at a time; later once I know which notes ratings I want to look at and use.
# pre-process to make smaller (remove columns not needed); merge into single csv
notes_ratings_list = [notes_ratings0, notes_ratings1, notes_ratings2, notes_ratings3, notes_ratings4, notes_ratings5, notes_ratings6, notes_ratings7, notes_ratings8, notes_ratings9, notes_ratings10, notes_ratings11, notes_ratings12, notes_ratings13, notes_ratings14, notes_ratings15]

# dropping columns from Notes_Ratings I don't need
# Initialize an empty dictionary to store the processed DataFrames
processed_files = {}

# Loop through the files and process them
for i, file in enumerate(notes_ratings_list):
    # Define columns to drop
    columns = "helpful, notHelpful, helpfulInformative, helpfulEmpathetic, helpfulUniqueContext, notHelpfulOpinionSpeculationOrBias, notHelpfulOutdated, notHelpfulOffTopic"
    quoted_columns = [col.strip() for col in columns.split(",")]
    
    # Process the file
    notes_ratings_clean = file.drop(columns=quoted_columns, errors='ignore')
    notes_ratings_clean = notes_ratings_clean.astype(str)
    
    # Save the processed DataFrame in the dictionary with a unique key
    processed_files[f'file_{i}'] = notes_ratings_clean

In [ ]:
# skip for now
# read in userEnrollment file when/if needed
import pandas as pd

enrollment = pd.read_csv('~/Downloads/userEnrollment-00000.tsv', sep='\t')

In [ ]:
### Old bits of code:

In [ ]:
## Old code using a regular expression...but Field encouraged me to change to a simpler substring search (which I did above)
# Check for keywords in "summary" column (text of Notes) using a regexp

import re

# ensure all objects in summary column are strings
df['summary'] = df['summary'].astype(str)

# uses a regular expression to search for each keyword in the "summary" text
def find_keywords(text, keywords):
    found_keywords = [
        keyword for keyword in keywords 
        if re.search(rf'\b{re.escape(keyword)}\b', str(text), re.IGNORECASE)
    ]
    return found_keywords